In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn

In [2]:
df=pd.read_csv('D:/NIH chest X-Ray/Data_Entry_2017.csv')

In [3]:
df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [4]:
df.drop('Follow-up #',axis=1)

,Image Index,Finding Labels,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,3,81,F,PA,2582,2991,0.143,0.143,NaN
...,...,...,...,...,...,...,...,...,...,...,...
112115,00030801_001.png,Mass|Pneumonia,30801,39,M,PA,2048,2500,0.168,0.168,NaN
112116,00030802_000.png,No Finding,30802,29,M,PA,2048,2500,0.168,0.168,NaN
112117,00030803_000.png,No Finding,30803,42,F,PA,2048,2500,0.168,0.168,NaN
112118,00030804_000.png,No Finding,30804,30,F,PA,2048,2500,0.168,0.168,NaN


In [5]:
df=df.drop(['Patient Age','Patient Gender','View Position','OriginalImage[Width','Height]'],axis=1)

In [6]:
df

,Image Index,Finding Labels,Follow-up #,Patient ID,OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,0.143,0.143,NaN
...,...,...,...,...,...,...,...
112115,00030801_001.png,Mass|Pneumonia,1,30801,0.168,0.168,NaN
112116,00030802_000.png,No Finding,0,30802,0.168,0.168,NaN
112117,00030803_000.png,No Finding,0,30803,0.168,0.168,NaN
112118,00030804_000.png,No Finding,0,30804,0.168,0.168,NaN


In [7]:
df=df.drop(['Follow-up #','OriginalImagePixelSpacing[x','y]','Unnamed: 11'],axis=1)

In [8]:
df

,Image Index,Finding Labels,Patient ID
0,00000001_000.png,Cardiomegaly,1
1,00000001_001.png,Cardiomegaly|Emphysema,1
2,00000001_002.png,Cardiomegaly|Effusion,1
3,00000002_000.png,No Finding,2
4,00000003_000.png,Hernia,3
...,...,...,...
112115,00030801_001.png,Mass|Pneumonia,30801
112116,00030802_000.png,No Finding,30802
112117,00030803_000.png,No Finding,30803
112118,00030804_000.png,No Finding,30804


In [9]:
df["Finding Labels"]=df["Finding Labels"].str.split("|")

In [10]:
type(df["Finding Labels"])

pandas.Series

In [11]:
df

,Image Index,Finding Labels,Patient ID
0,00000001_000.png,[Cardiomegaly],1
1,00000001_001.png,"[Cardiomegaly, Emphysema]",1
2,00000001_002.png,"[Cardiomegaly, Effusion]",1
3,00000002_000.png,[No Finding],2
4,00000003_000.png,[Hernia],3
...,...,...,...
112115,00030801_001.png,"[Mass, Pneumonia]",30801
112116,00030802_000.png,[No Finding],30802
112117,00030803_000.png,[No Finding],30803
112118,00030804_000.png,[No Finding],30804


In [12]:
labels=df['Finding Labels'].explode()
labels

0         Cardiomegaly
1         Cardiomegaly
1            Emphysema
2         Cardiomegaly
2             Effusion
              ...     
112115       Pneumonia
112116      No Finding
112117      No Finding
112118      No Finding
112119      No Finding
Name: Finding Labels, Length: 141537, dtype: str

In [13]:
unique_labels=labels.unique()
type(unique_labels)

pandas.arrays.StringArray

In [14]:
python_list=[]
for index, label in enumerate(unique_labels):
    python_list.append(label)

python_list.remove('No Finding')


In [15]:
label_to_index={}
for index, label in enumerate(python_list):
    label_to_index[label]=index

label_to_index

{'Cardiomegaly': 0,
 'Emphysema': 1,
 'Effusion': 2,
 'Hernia': 3,
 'Infiltration': 4,
 'Mass': 5,
 'Nodule': 6,
 'Atelectasis': 7,
 'Pneumothorax': 8,
 'Pleural_Thickening': 9,
 'Pneumonia': 10,
 'Fibrosis': 11,
 'Edema': 12,
 'Consolidation': 13}

In [21]:
import json

disease_mapping = {
    str(i): disease 
    for i, disease in enumerate(label_to_index)
}

with open("disease_mapping.json", "w") as f:
    json.dump(disease_mapping, f, indent=4)

In [16]:

def encode_labels(row_labels):
    vector = [0] * len(label_to_index)
    
    for label in row_labels:
        if label=='No Finding':
            continue
        else:
            idx=label_to_index[label]
            vector[idx]=1
    
    return vector

In [17]:
df['Labels_encoded']=df['Finding Labels'].apply(encode_labels)


In [20]:
df.to_csv('processed_data_entry.csv', index=False)

In [19]:
df

,Image Index,Finding Labels,Patient ID,Labels_encoded
0,00000001_000.png,[Cardiomegaly],1,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,00000001_001.png,"[Cardiomegaly, Emphysema]",1,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,00000001_002.png,"[Cardiomegaly, Effusion]",1,"[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,00000002_000.png,[No Finding],2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,00000003_000.png,[Hernia],3,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
...,...,...,...,...
112115,00030801_001.png,"[Mass, Pneumonia]",30801,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0]"
112116,00030802_000.png,[No Finding],30802,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
112117,00030803_000.png,[No Finding],30803,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
112118,00030804_000.png,[No Finding],30804,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
